# Predicting Links with Graph Neural Networks

## 실행 환경 셋업 (어느 머신에서든)

레포 루트의 `requirements.txt` 하나로 재현됩니다. CUDA(NVIDIA GPU) 없이 CPU로 전부 실행 가능.

```bash
cd <repo-root>
python3.10 -m venv .venv          # 처음 한 번
.venv/bin/pip install -r requirements.txt
.venv/bin/python -m ipykernel install --user --name gnn-2026 --display-name "Python (.venv gnn-2026)"
```

- **데이터**: Cora는 첫 실행 시 이 폴더(`practice/Cora/`)에 자동 다운로드됨 (git ignore 처리, 인터넷 필요)
- **소요 시간(CPU 기준)**: VGAE 학습(300 epoch) 약 1~2분 / SEAL 서브그래프 전처리 수 분 + DGCNN 학습(30 epoch) 약 10분 내외
- **참고**: torch-scatter 등 컴파일 휠은 이 노트북에 불필요 (첫 코드 셀의 pip install은 스킵 처리됨)
- ⚠️ Apple Silicon에서 numpy/torch import 시 `incompatible architecture (x86_64)` 에러가 나면 venv가 Intel 휠로 만들어진 것 — `.venv/bin/pip install --force-reinstall -r requirements.txt`로 해결

기록된 출력(Test AUC 0.8691/0.786 등)은 슬라이드가 인용하는 기준 실행 결과이므로, 보존하려면 재실행 전에 사본을 뜨세요.

In [1]:
import torch
# [skipped] !pip install -q torch-scatter~=2.1.0 torch-sparse~=0.6.16 torch-cluster~=1.6.0 torch-spline-conv~=1.2.1 torch-geometric==2.2.0 -f https://data.pyg.org/whl/torch-{torch.__version__}.html

torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### 이 챕터를 꿰뚫는 한 줄

> **"링크 예측은 GNN이 자연스럽게 풀 수 없는 anchored 과제다. VGAE는 노드 feature에 그 일을 떠넘기고, SEAL은 라벨링으로 직접 해결한다."**

암묵적 전제 하나를 깔고 시작한다: **지금 보이는 그래프는 표본이다.** 참 엣지 집합 $E^*$가 있고 관측 $E_{\text{obs}} \subset E^*$는 부분 관측. "없는 엣지"로 보이는 쌍은 (a) 진짜 비엣지일 수도 (b) 엣지인데 결측일 수도 있다 — 확정 음성 라벨이 없는 **PU-learning**(Positive-Unlabeled) 구조다. 벤치마크(Cora)는 정답 그래프에서 엣지를 균등 무작위로 일부러 숨기는 시뮬레이션이라는 점도 기억해두자.

## Graph Autoencoder (VAE) and Variational Graph Autoencoder (VGAE)

### GAE의 아이디어: 인코딩 + 내적 디코딩

![GAE decoder](textbook_images/ch10_fig02_gae_decoder_AeqZTZ.png)

*(Labonne, 2023)*

$$\mathbf{Z} = \mathrm{GCN}(X, A), \qquad \hat{A} = \sigma(\mathbf{Z}\mathbf{Z}^\top)$$

**계보**: GAE(Graph Autoencoder)는 seq2seq(번역) 갈래가 아니라 **오토인코더 갈래** — 정확히는 VAE(Variational Autoencoder)의 그래프 이식(Kipf & Welling, 2016)이다. 이식에서 바뀐 두 지점:

1. 복원 대상이 **인접행렬 $A$만** — 피처 $X$는 재료로만 쓴다
2. 디코더가 **학습 없는 내적** — VAE 디코더는 보통 신경망

**압축이 곧 예측**: 병목 $Z$($N\times 16$)는 $A$($N\times N$, Cora 기준 약 366만 쌍)를 외울 수 없다. 그래서 연결의 *규칙성*(homophily)을 학습할 수밖에 없고, 그 규칙으로 "있어야 할 자리"에 높은 점수를 준다 — 관측 안 된 쌍의 예측이 복원에서 공짜로 나온다.

**내적 디코더가 강제하는 가정**: $\sigma(\mathbf{z}_u^\top\mathbf{z}_v)$ 한 줄 = homophily 가설("가까우면 연결된다"). heterophily 그래프(사기 탐지), 방향 그래프(내적은 항상 대칭), 글로벌 구조에서는 깨진다. Cora는 homophily가 강해 유리한 무대.

**GAE → VGAE**: 임베딩을 점이 아니라 **가우시안 분포** $q(\mathbf{z}_i|X,A) = \mathcal{N}(\boldsymbol{\mu}_i, \mathrm{diag}(\boldsymbol{\sigma}_i^2))$로 출력. 손실에 KL 항이 붙고($\frac{1}{N}$ 가중치), 생성 모델 관점 $A_{uv} \sim \mathrm{Bernoulli}(\sigma(\mathbf{z}_u^\top\mathbf{z}_v))$이 완성된다 — "그래프는 표본" 프레이밍의 형식화. 실용적으로 V는 정규화 트릭에 가깝다.

In [2]:
import numpy as np
np.random.seed(0)
import torch
torch.manual_seed(0)
import matplotlib.pyplot as plt
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = T.Compose([
    T.NormalizeFeatures(),
    T.ToDevice(device),
    T.RandomLinkSplit(num_val=0.05, num_test=0.1, is_undirected=True, split_labels=True, add_negative_train_samples=False),
])

dataset = Planetoid('.', name='Cora', transform=transform)

train_data, val_data, test_data = dataset[0]

Processing...
Done!


### 방금 일어난 일: RandomLinkSplit

- 노드 분류와 달리 **엣지를** train/val/test로 나눈다. test/val 엣지는 **메시지 패싱 그래프에서도 빠진다** (누설 방지).
- 음성은 "전수 라벨링"이 아니라 **샘플링**: val/test 음성은 양성과 같은 수로 뽑아 고정, train 음성은 (`add_negative_train_samples=False`라서) 매 epoch `recon_loss` 안에서 재샘플링.
- 평가지표는 **AUC**(Area Under the ROC Curve)와 **AP**(Average Precision) — 확정 음성 라벨이 없고 임계값은 응용마다 다르니 **순위 기반** 지표를 쓴다.
  - AUC = 무작위 양성·음성 쌍 대결에서 양성이 이길 확률 (0.5 = 무작위, 1.0 = 완벽)
  - AP = 점수순으로 읽으며 양성마다 그 시점 precision을 평균 — 상위권 오염에 민감

In [3]:
from torch_geometric.nn import GCNConv, VGAE

class Encoder(torch.nn.Module):
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.conv1 = GCNConv(dim_in, 2 * dim_out)
        self.conv_mu = GCNConv(2 * dim_out, dim_out)
        self.conv_logstd = GCNConv(2 * dim_out, dim_out)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

### 인코더 구조 읽기

GCN(Graph Convolutional Network) 레이어가 **세 개**: 공유 `conv1` 뒤에 `conv_mu`와 `conv_logstd`가 갈라져 각각 $\boldsymbol{\mu}$와 $\log\boldsymbol{\sigma}$를 출력한다 (1433 → 32 → 16×2). 학습 시 `VGAE` 래퍼가 reparameterization trick $\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}$으로 샘플링한다. 디코더는 파라미터가 없으므로 정의할 것도 없다 — **학습은 전부 인코더에서 일어난다.**

In [4]:
model = VGAE(Encoder(dataset.num_features, 16)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def train():
    model.train()
    optimizer.zero_grad()
    z = model.encode(train_data.x, train_data.edge_index)
    loss = model.recon_loss(z, train_data.pos_edge_label_index) + (1 / train_data.num_nodes) * model.kl_loss()
    loss.backward()
    optimizer.step()
    return float(loss)

@torch.no_grad()
def test(data):
    model.eval()
    z = model.encode(data.x, data.edge_index)
    return model.test(z, data.pos_edge_label_index, data.neg_edge_label_index)

for epoch in range(301):
    loss = train()
    val_auc, val_ap = test(test_data)
    if epoch % 50 == 0:
        print(f'Epoch {epoch:>2} | Loss: {loss:.4f} | Val AUC: {val_auc:.4f} | Val AP: {val_ap:.4f}') 

test_auc, test_ap = test(test_data) 
print(f'Test AUC: {test_auc:.4f} | Test AP {test_ap:.4f}')

Epoch  0 | Loss: 3.4775 | Val AUC: 0.6784 | Val AP: 0.7202


Epoch 50 | Loss: 1.3314 | Val AUC: 0.6954 | Val AP: 0.7249


Epoch 100 | Loss: 1.1789 | Val AUC: 0.7198 | Val AP: 0.7284


Epoch 150 | Loss: 1.1027 | Val AUC: 0.7636 | Val AP: 0.7503


Epoch 200 | Loss: 1.0124 | Val AUC: 0.8360 | Val AP: 0.8319


Epoch 250 | Loss: 0.9814 | Val AUC: 0.8482 | Val AP: 0.8475


Epoch 300 | Loss: 0.9639 | Val AUC: 0.8691 | Val AP: 0.8722
Test AUC: 0.8691 | Test AP 0.8722


### 학습 루프에서 주목할 두 가지

1. `recon_loss(z, pos_edge_label_index)` — 양성만 넘겨도 내부에서 **음성 엣지를 매 epoch 자동 재샘플링**해 BCE(Binary Cross-Entropy)를 계산한다.
2. `(1/N) * kl_loss()` — 표준 ELBO에는 없는 $\frac{1}{N}$. 노드 수에 비례해 쌓이는 KL을 노드당 평균으로 만든 경험적 선택이고, β-VAE 관점에서 "복원 우선" 가중치다.

기준 실행 결과: **Test AUC 0.869 / AP 0.872** (책: 0.88 / 0.90 — 거의 일치).

In [5]:
z = model.encode(test_data.x, test_data.edge_index) 
Ahat = torch.sigmoid(z @ z.T)
Ahat

tensor([[0.8228, 0.3109, 0.7193,  ..., 0.4668, 0.7864, 0.7889],
        [0.3109, 0.9431, 0.7427,  ..., 0.6726, 0.5633, 0.6142],
        [0.7193, 0.7427, 0.8340,  ..., 0.5885, 0.8100, 0.8303],
        ...,
        [0.4668, 0.6726, 0.5885,  ..., 0.5713, 0.5586, 0.5583],
        [0.7864, 0.5633, 0.8100,  ..., 0.5586, 0.8502, 0.8381],
        [0.7889, 0.6142, 0.8303,  ..., 0.5583, 0.8381, 0.8489]],
       grad_fn=<SigmoidBackward0>)

### $\hat{A}$가 보여주는 것: homophily를 학습했다

![Ahat heatmap](textbook_images/ch10_runtime_vgae_ahat_heatmap.png)

$\hat{A} = \sigma(ZZ^\top)$를 클래스순으로 정렬해 그리면 **블록 구조**가 나타난다 — 같은 클래스 노드끼리 높은 점수. 내적 디코더의 homophily 가정이 데이터에서 실제로 실현된 모습이다.

![t-SNE](textbook_images/ch10_runtime_vgae_embedding_tsne.png)

t-SNE 투영을 보면 **링크 예측만 학습했는데** Cora의 7개 클래스가 어느 정도 군집화되어 있다 — homophily 그래프에서 링크와 클래스는 강하게 결합되어 있다는 증거. (두 그림 모두 `make_figures.py`로 생성)

## SEAL

### 왜 GNN만으로는 안 되는가: 1-WL 한계

4-cycle(정사각형 그래프)에서 모든 노드는 차수 2, 이웃 모양 동일 — feature까지 같으면 GNN은 $\mathbf{z}_1 = \mathbf{z}_2 = \mathbf{z}_3 = \mathbf{z}_4$를 준다. 그러면 내적 디코더는 엣지 $(1,2)$와 비엣지 $(1,3)$에 **같은 점수**를 줄 수밖에 없다. 학습 부족이 아니라 **1-WL(Weisfeiler-Lehman) 표현력의 구조적 한계**다(Xu et al., 2019). 처방 = 노드 feature에 "타깃이 누구인지"를 박아넣는 **labeling trick**.

### SEAL의 출발점: 링크 예측 → 서브그래프 분류

![SEAL pipeline](textbook_images/ch10_fig03_seal_pipeline.png)

*(Labonne, 2023)*

핵심 통찰: 휴리스틱(CN(Common Neighbors), AA(Adamic–Adar), Katz)이 전부 **enclosing subgraph의 함수**다. 그렇다면 어떤 휴리스틱을 쓸지 미리 정하지 말고, $k$-hop enclosing subgraph 전체를 GNN에 주고 최적의 점수함수를 *학습*하게 하자 — 링크 예측이 **이진 그래프 분류**로 환원된다. γ-decaying 정리(Zhang & Chen, 2018)가 표현력을 보장.

![k-hop](textbook_images/ch10_fig01_khop_neighborhood.png)

*(Labonne, 2023)* — $k$-hop 이웃: 최대 $k$개 엣지로 도달 가능한 노드 집합. SEAL은 두 타깃의 $k$-hop 합집합을 쓰며 $k=2$가 표준.

In [6]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.sparse.csgraph import shortest_path

import torch.nn.functional as F
from torch.nn import Conv1d, MaxPool1d, Linear, Dropout, BCEWithLogitsLoss

from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, aggr
from torch_geometric.utils import k_hop_subgraph, to_scipy_sparse_matrix

In [7]:
# Load Cora dataset
transform = RandomLinkSplit(num_val=0.05, num_test=0.1, is_undirected=True, split_labels=True)
dataset = Planetoid('.', name='Cora', transform=transform)
train_data, val_data, test_data = dataset[0]
train_data

Data(x=[2708, 1433], edge_index=[2, 8976], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], pos_edge_label=[4488], pos_edge_label_index=[2, 4488], neg_edge_label=[4488], neg_edge_label_index=[2, 4488])

### `seal_processing` 읽는 법: DRNL과 누설 방지 3종

**DRNL**(Double-Radius Node Labeling): 각 노드의 좌표 $(d_x, d_y)$(두 타깃까지의 거리)를 충돌 없이 정수 하나로 접는 perfect hash다.

| $(d_x,d_y)$ | $z$ | 의미 |
|---|---|---|
| 타깃 | **1** | $u,v$ 본인 |
| $(1,1)$ | **2** | 공통 이웃 |
| $(1,2)$ | 3 | |
| $(1,3)$ / $(2,2)$ | 4 / 5 | |

공식 자체는 아무래도 좋다 — 필요한 성질은 단사성뿐이고, 진짜 발명은 **라벨이 타깃 쌍에 의존한다**는 발상(labeling trick). $z{=}2$ 개수 세기 = CN, $z{=}2$ 노드의 차수까지 보면 AA — 휴리스틱이 라벨의 함수로 자동 포섭된다.

**누설(leakage) 방지 3종** — 셋 다 "모델이 정답의 흔적을 보지 못하게 한다"는 같은 원칙:

1. **타깃 엣지 제거** (`mask1 & mask2`, 양방향): 안 하면 "엣지가 보이네 → 1"로 답을 베낌 — 학습 AUC만 완벽해지고 일반화 0.
2. **거리 계산 시 상대 노드 제외** (`adj_wo_dst`로 `d_src` 계산): 상대 타깃을 거쳐가는 지름길이 거리를 왜곡하는 걸 막고 **진짜 우회 경로** 길이를 측정.
3. **양성·음성 동일 절차**: 분포가 달라지면 그 차이로 치팅한다.

![DRNL example](textbook_images/ch10_runtime_drnl_example.png)

실제 Cora 서브그래프의 DRNL 라벨링 — 가운데 $z{=}2$ 노드가 공통 이웃, $z{=}0$은 상대 타깃 제거 시 도달 불가 노드.

In [8]:
def seal_processing(dataset, edge_label_index, y):
    data_list = []

    for src, dst in edge_label_index.t().tolist():
        sub_nodes, sub_edge_index, mapping, _ = k_hop_subgraph([src, dst], 2, dataset.edge_index, relabel_nodes=True)
        src, dst = mapping.tolist()

        # Remove target link from the subgraph
        mask1 = (sub_edge_index[0] != src) | (sub_edge_index[1] != dst)
        mask2 = (sub_edge_index[0] != dst) | (sub_edge_index[1] != src)
        sub_edge_index = sub_edge_index[:, mask1 & mask2]

        # Double-radius node labeling (DRNL)
        src, dst = (dst, src) if src > dst else (src, dst)
        adj = to_scipy_sparse_matrix(sub_edge_index, num_nodes=sub_nodes.size(0)).tocsr()

        idx = list(range(src)) + list(range(src + 1, adj.shape[0]))
        adj_wo_src = adj[idx, :][:, idx]

        idx = list(range(dst)) + list(range(dst + 1, adj.shape[0]))
        adj_wo_dst = adj[idx, :][:, idx]

        # Calculate the distance between every node and the source target node
        d_src = shortest_path(adj_wo_dst, directed=False, unweighted=True, indices=src)
        d_src = np.insert(d_src, dst, 0, axis=0)
        d_src = torch.from_numpy(d_src)

        # Calculate the distance between every node and the destination target node
        d_dst = shortest_path(adj_wo_src, directed=False, unweighted=True, indices=dst-1)
        d_dst = np.insert(d_dst, src, 0, axis=0)
        d_dst = torch.from_numpy(d_dst)

        # Calculate the label z for each node
        dist = d_src + d_dst
        z = 1 + torch.min(d_src, d_dst) + dist // 2 * (dist // 2 + dist % 2 - 1)
        z[src], z[dst], z[torch.isnan(z)] = 1., 1., 0.
        z = z.to(torch.long)

        # Concatenate node features and one-hot encoded node labels (with a fixed number of classes)
        node_labels = F.one_hot(z, num_classes=200).to(torch.float)
        node_emb = dataset.x[sub_nodes]
        node_x = torch.cat([node_emb, node_labels], dim=1)

        # Create data object
        data = Data(x=node_x, z=z, edge_index=sub_edge_index, y=y)
        data_list.append(data)

    return data_list

In [9]:
# Enclosing subgraphs extraction
train_pos_data_list = seal_processing(train_data, train_data.pos_edge_label_index, 1)
train_neg_data_list = seal_processing(train_data, train_data.neg_edge_label_index, 0)

val_pos_data_list = seal_processing(val_data, val_data.pos_edge_label_index, 1)
val_neg_data_list = seal_processing(val_data, val_data.neg_edge_label_index, 0)

test_pos_data_list = seal_processing(test_data, test_data.pos_edge_label_index, 1)
test_neg_data_list = seal_processing(test_data, test_data.neg_edge_label_index, 0)

In [10]:
train_dataset = train_pos_data_list + train_neg_data_list
val_dataset = val_pos_data_list + val_neg_data_list
test_dataset = test_pos_data_list + test_neg_data_list

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

### DGCNN(Deep Graph CNN): 왜 SortPooling + Conv1D인가

서브그래프 크기가 매번 다르므로 **고정 길이 표현**이 필요하다. sum/mean pooling도 가능하지만 노드 간 상대 순서 정보를 전부 버린다. **SortPooling**은 마지막 GCN 채널 값으로 노드를 일관된 순서로 정렬해 상위 $k{=}30$개만 취한다(부족하면 0-padding) — 그래프가 "단어 순서 있는 문장"처럼 되고, 그 시퀀스를 Conv1D로 처리한다.

> SEAL이 링크 예측을 그래프 분류로 환원했듯, DGCNN은 그래프를 시퀀스로 환원한다 — 환원의 연쇄.

In [11]:
class DGCNN(torch.nn.Module):
    def __init__(self, dim_in, k=30):
        super().__init__()

        # GCN layers
        self.gcn1 = GCNConv(dim_in, 32)
        self.gcn2 = GCNConv(32, 32)
        self.gcn3 = GCNConv(32, 32)
        self.gcn4 = GCNConv(32, 1)

        # Global sort pooling
        self.global_pool = aggr.SortAggregation(k=k)

        # Convolutional layers
        self.conv1 = Conv1d(1, 16, 97, 97)
        self.conv2 = Conv1d(16, 32, 5, 1)
        self.maxpool = MaxPool1d(2, 2)

        # Dense layers
        self.linear1 = Linear(352, 128)
        self.dropout = Dropout(0.5)
        self.linear2 = Linear(128, 1)

    def forward(self, x, edge_index, batch):
        # 1. Graph Convolutional Layers
        h1 = self.gcn1(x, edge_index).tanh()
        h2 = self.gcn2(h1, edge_index).tanh()
        h3 = self.gcn3(h2, edge_index).tanh()
        h4 = self.gcn4(h3, edge_index).tanh()
        h = torch.cat([h1, h2, h3, h4], dim=-1)

        # 2. Global sort pooling
        h = self.global_pool(h, batch)

        # 3. Traditional convolutional and dense layers
        h = h.view(h.size(0), 1, h.size(-1))
        h = self.conv1(h).relu()
        h = self.maxpool(h)
        h = self.conv2(h).relu()
        h = h.view(h.size(0), -1)
        h = self.linear1(h).relu()
        h = self.dropout(h)
        h = self.linear2(h).sigmoid()

        return h

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DGCNN(train_dataset[0].num_features).to(device)
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.0001)
criterion = BCEWithLogitsLoss()

def train():
    model.train()
    total_loss = 0

    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = criterion(out.view(-1), data.y.to(torch.float))
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * data.num_graphs

    return total_loss / len(train_dataset)

@torch.no_grad()
def test(loader):
    model.eval()
    y_pred, y_true = [], []

    for data in loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.batch)
        y_pred.append(out.view(-1).cpu())
        y_true.append(data.y.view(-1).cpu().to(torch.float))

    auc = roc_auc_score(torch.cat(y_true), torch.cat(y_pred))
    ap = average_precision_score(torch.cat(y_true), torch.cat(y_pred))

    return auc, ap

for epoch in range(31):
    loss = train()
    val_auc, val_ap = test(val_loader)
    print(f'Epoch {epoch:>2} | Loss: {loss:.4f} | Val AUC: {val_auc:.4f} | Val AP: {val_ap:.4f}')

test_auc, test_ap = test(test_loader)
print(f'Test AUC: {test_auc:.4f} | Test AP {test_ap:.4f}')

Epoch  0 | Loss: 0.7022 | Val AUC: 0.7457 | Val AP: 0.7723


Epoch  1 | Loss: 0.6403 | Val AUC: 0.7928 | Val AP: 0.8340


Epoch  2 | Loss: 0.6106 | Val AUC: 0.7959 | Val AP: 0.8423


Epoch  3 | Loss: 0.6060 | Val AUC: 0.7972 | Val AP: 0.8434


Epoch  4 | Loss: 0.6038 | Val AUC: 0.7970 | Val AP: 0.8443


...

Epoch 27 | Loss: 0.5868 | Val AUC: 0.7978 | Val AP: 0.8274


Epoch 28 | Loss: 0.5872 | Val AUC: 0.7940 | Val AP: 0.8268


Epoch 29 | Loss: 0.5866 | Val AUC: 0.7991 | Val AP: 0.8292


Epoch 30 | Loss: 0.5858 | Val AUC: 0.7981 | Val AP: 0.8302


Test AUC: 0.7860 | Test AP 0.8022


### 결과 해석과 정리

기준 실행: **Test AUC 0.786 / AP 0.802** (책: 0.89 / 0.90). 낮은 원인 추정 — ① 30 epoch이 DGCNN에 짧음 ② `BCEWithLogitsLoss`를 sigmoid 출력에 적용(이중 sigmoid, 책 코드의 알려진 이슈) ③ 음성 샘플링의 시드 의존. *재현성 자체가 메타 교훈.*

| | **VGAE** | **SEAL** |
|---|---|---|
| 점수함수 | $\sigma(\mathbf{z}_u^\top\mathbf{z}_v)$ — 고정 내적 | 학습된 GNN 분류기 |
| 쌍 관계 정보 | 간접 (임베딩 유사도) | 직접 (DRNL로 구조 명시) |
| 숨은 가정 | homophily·대칭 | 가정을 데이터에서 학습 |
| 새 노드/그래프 | transductive — 재학습 | inductive — 즉시 추론 |
| 비용 | encode 1회 후 모든 쌍 저렴 | 쌍마다 서브그래프 추출 |

Cora는 homophily 강함 + feature 풍부 + 소규모 + 무방향이라 **두 모델 모두에게 후한 무대** — 비슷한 점수는 착시고, 가정과 구조는 전혀 다른 모델이다.